In [3]:
import langchain


In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)


True

In [5]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

Example-1 : Simple LLM Call with Streaming

In [6]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

In [7]:
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002335A970590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002335A9712B0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

## Create messages


In [8]:
messages = [
    SystemMessage("You are a helpful AI Assistant "),
    HumanMessage("What are the top 2 benefits of using Langchain?")
]

In [9]:
## invoke the model
response=model.invoke(messages)
response

AIMessage(content='**Top 2 benefits of using LangChain**\n\n1. **Modular, composable architecture**  \n   - LangChain breaks down LLM‑driven workflows into reusable, interchangeable components (prompts, memory, tools, callbacks, etc.).  \n   - This lets you quickly prototype, test, and iterate on complex chains—adding or swapping out parts without rewriting the whole app.  \n   - The composability also makes it easy to share building blocks across projects, speeding up development and reducing duplication.\n\n2. **Seamless integration with external data, services, and tools**  \n   - LangChain ships with pre‑built connectors for APIs, databases, web scraping, vector stores, and many LLM providers.  \n   - You can extend an LLM’s capabilities by chaining it with these tools (e.g., retrieving facts from a knowledge base, calling a calculator, or updating a database) in a single, coherent pipeline.  \n   - This tight integration turns a vanilla LLM into a full‑stack application that can r

In [10]:
print(response.content)

**Top 2 benefits of using LangChain**

1. **Modular, composable architecture**  
   - LangChain breaks down LLM‑driven workflows into reusable, interchangeable components (prompts, memory, tools, callbacks, etc.).  
   - This lets you quickly prototype, test, and iterate on complex chains—adding or swapping out parts without rewriting the whole app.  
   - The composability also makes it easy to share building blocks across projects, speeding up development and reducing duplication.

2. **Seamless integration with external data, services, and tools**  
   - LangChain ships with pre‑built connectors for APIs, databases, web scraping, vector stores, and many LLM providers.  
   - You can extend an LLM’s capabilities by chaining it with these tools (e.g., retrieving facts from a knowledge base, calling a calculator, or updating a database) in a single, coherent pipeline.  
   - This tight integration turns a vanilla LLM into a full‑stack application that can read, reason, and act on real‑

In [11]:
## Stream Example
for chunk in model.stream(messages):
    print(chunk.content, end="",flush=True)

**Top 2 benefits of using LangChain**

| # | Benefit | Why it matters |
|---|---------|----------------|
| **1. Rapid, modular development** | LangChain provides a library of pre‑built, composable “chains” (e.g., prompt templates, LLM wrappers, vector‑store connectors, memory modules). You can assemble or swap components with a few lines of code, dramatically cutting the time from idea to working prototype. |
| **2. Seamless orchestration of complex LLM workflows** | LangChain lets you combine LLM calls, retrieval‑augmented generation, external APIs, and custom logic into a single pipeline. This makes it easy to build sophisticated, stateful applications—like question‑answering agents, summarizers, or multi‑step reasoning—without reinventing the integration plumbing. |

Together, these advantages mean you can prototype quickly, then scale to production‑ready, feature‑rich AI applications with minimal boilerplate.

## Dynamic Prompt Template

In [13]:
from langchain_core.prompts import ChatPromptTemplate
## create translation app
translation_template=ChatPromptTemplate.from_messages([
    ("system","You are a Professional Translator. Translate the following text from {source_language} to {target_language}. Maintain the tone and style."),
    ("user","{text}")
])
## Using the messages 
prompt=translation_template.invoke({
    "source_language":"English",
    "target_language":"French",
    "text":"Hello, how are you?"
})
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a Professional Translator. Translate the following text from English to French. Maintain the tone and style.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})])

In [14]:
translated_response=model.invoke(prompt)
print(translated_response.content)

Salut, comment vas‑tu ?


## Building you first chain


In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def create_story_chain():
    ## Template for story generation
    story_template = ChatPromptTemplate.from_messages([
        ("system", "You are a story writer. Write a story based on the following prompt. The story should be engaging and suitable for all ages."),
        ("user", "{story_prompt}")
    ])

    ## Template for story summary
    summary_template = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Summarize the following story in 2-3 sentences."),
        ("user", "{story}")
    ])

    ## Chain: prompt -> model -> text
    story_chain = story_template | model | StrOutputParser()

    ## Pehle story banao, phir uski summary
    full_chain = (
        story_chain
        | (lambda story: {"story": story})
        | summary_template
        | model
        | StrOutputParser()
    )
    return full_chain, story_chain

In [17]:
full_chain, story_chain = create_story_chain()

story = story_chain.invoke({"story_prompt": "A brave little robot who loves gardening"})
print(story)

**Sprout the Brave Little Robot**

In the heart of the city, tucked between a bakery and a music shop, was a green patch of earth that everyone called the “Community Garden.” It was a place where children planted carrots, adults grew tomatoes, and the air smelled of damp soil and sunshine. But the most special thing in that garden was a tiny, shiny robot named Sprout.

Sprout was not like the big, clunky machines that helped build bridges or haul cargo. He was small—just the size of a shoebox—and covered in a bright green paint that matched the leaves he loved so much. His head was a clear dome that could see the world in all its colors, and his arms were slender, perfect for digging and pruning.

Every morning, Sprout would buzz to life with a cheerful “Beep!” and begin his gardening routine. He’d slide a little watering can across the soil, spraying droplets that glittered like tiny stars. He’d use his sensor arm to feel the earth and tell the seedlings when they needed more or less 

In [18]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

story_template = ChatPromptTemplate.from_messages([
    ("system", "You are a story writer. Write a short story based on the following prompt. The story should be engaging and suitable for all ages."),
    ("user", "{story_prompt}")
])

summary_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Summarize the following story in 2-3 sentences."),
    ("user", "{story}")
])

# Pipe: prompt -> model -> text, phir story ke saath summary bhi
connected_chain = (
    {"story": story_template | model | StrOutputParser()}
    | RunnablePassthrough.assign(
        summary=summary_template | model | StrOutputParser()
    )
)

result = connected_chain.invoke({"story_prompt": "A brave little robot who loves gardening"})

print("STORY:\n", result["story"])
print("\nSUMMARY:\n", result["summary"])

STORY:
 **Sprout the Garden‑Bot**

In the little town of Willowbrook, where the streets were lined with cobblestones and the air smelled of fresh bread, there lived a very special robot named Sprout. Sprout was no larger than a loaf of bread and was covered in a shiny silver shell with a bright green patch that looked like a leaf. His eyes were two small, glowing LEDs that twinkled whenever he was excited.

Sprout’s greatest joy was gardening. Every morning, before the sun climbed over the hills, he would tip his tiny metal head and say, “Good day, sunflowers!” He had a tiny watering can that was part of his arm, a little sprayer that could sprinkle water in a fine mist, and a set of tiny pruning shears that glimmered like a pair of scissors. He loved the way the soil smelled when it was damp and the way the plants leaned toward the light.

One bright Saturday, Sprout was busy arranging his garden. He had planted rows of tomatoes, a patch of basil, and a curious little row of violet fl